# GPU notebook 1: model-size selection

Trains the trees, the LSTM without attention and STAWarm (two layers, 128 units) on Azure 2021 (three seeds) and
Huawei day 30, then runs STAWarm at 256 and 64 units on seed 0. The validation losses printed during training
are used to choose the model size for notebook 2; test-period results of this notebook are not used for that choice.

**Setup.** Upload the repository (with `src/`, `scripts/`, `tests/` and `pyproject.toml`) as a Kaggle dataset, and the
three files of `data/processed/` as a second dataset; attach both; accelerator GPU (T4); Internet can be off.
Then *Save Version → Save & Run All*.

In [ ]:
# 1. Find the code and data (any depth under /kaggle/input). No pip install: the code is put on Python's path.
import glob, os, shutil, sys
print('looking for the code ...', flush=True)
hits = [p for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), 'src', 'stawarm'))]
print('found:', hits, flush=True)
assert hits, 'Code not found. Check that the stawarm-code dataset is attached (Add Input) and contains pyproject.toml and src/.'
# Use the copy of this repository (it has reproduce.py); older copies in the same dataset are ignored.
current = [p for p in hits if os.path.exists(os.path.join(os.path.dirname(p), 'reproduce.py'))]
assert current, 'The attached stawarm-code dataset holds an older version of the code; upload the current repository as a new version.'
code_dir = os.path.dirname(current[0])
WORK = '/kaggle/working/stawarm'
if not os.path.exists(WORK):
    shutil.copytree(code_dir, WORK)
os.chdir(WORK)
# The repository ships the paper's result files in results/; keep them aside for comparison so that this
# notebook writes only its own outputs to results/.
if os.path.isdir('results') and not os.path.isdir('results_paper'):
    shutil.move('results', 'results_paper')
os.makedirs('results', exist_ok=True)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')          # used by every !python command below
sys.path.insert(0, os.path.join(WORK, 'src'))
os.makedirs('data/processed', exist_ok=True)
for name in ['azure2021_events.parquet', 'huawei_r2_day30_events.parquet', 'huawei_r2_cold_start_stats.parquet']:
    src = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    print(name, '->', src, flush=True)
    assert src, f'{name} not found: check that the stawarm-data dataset is attached'
    if not os.path.exists(f'data/processed/{name}'):
        os.symlink(src[0], f'data/processed/{name}')
import stawarm
print('code from', code_dir, '| package OK:', stawarm.__file__)
print('data:', os.listdir('data/processed'), flush=True)

In [ ]:
# 2. Record the software and hardware environment of this run
!python scripts/tools/record_environment.py results/environment_01_gpu_model_size_selection.txt

In [ ]:
# 3. Hardware check
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
!nproc; free -g | head -2

In [ ]:
# 4. Tests (should end with '35 passed')
!python -m pytest -q 2>&1 | tail -2

## 5. Azure 2021: three seeds (about 1.5 h each)

In [ ]:
for seed in [0, 1, 2]:
    !python -u scripts/experiments/run_policies.py --variants app_nolead,lstm_full,stawarm_full --risk-on none --lead-feature --thetas 0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9 --seed {seed} --tag size128_s{seed} 2>&1 | grep -v Warning

## 6. Huawei day 30 (about 1 h)

In [ ]:
!python -u scripts/experiments/run_policies.py --dataset huawei_day30 --variants app_nolead,lstm_full,stawarm_full --risk-on none --lead-feature --thetas 0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9 --tag size128 2>&1 | grep -v Warning

## 7. Summaries at matched memory (mean over seeds, Holm-corrected tests)

In [ ]:
!python scripts/analysis/summarize_runs.py "results/policies_azure2021_size128_s*" --out results/summary_size128_azure.txt
!python scripts/analysis/summarize_runs.py "results/policies_huawei_day30_size128*" --out results/summary_size128_huawei.txt

## 8. Optional: model-size check (seed 0, about 45 min)
Two further sizes for the validation-based size choice.

In [ ]:
RUN_SIZE_CHECK = True
if RUN_SIZE_CHECK:
    !python -u scripts/experiments/run_policies.py --variants stawarm_full --risk-on none --lead-feature --thetas 0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9 --hidden 256 --layers 2 --seed 0 --tag size256_s0 2>&1 | grep -v Warning
    !python -u scripts/experiments/run_policies.py --variants stawarm_full --risk-on none --lead-feature --thetas 0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9 --hidden 64 --layers 1 --seed 0 --tag size64_s0 2>&1 | grep -v Warning

In [ ]:
# 9. Compare this run with the paper's result files (zero differences expected for tree-based runs)
!python scripts/analysis/compare_results.py results results_paper --out results/comparison_01_gpu_model_size_selection.txt

In [ ]:
# 10. Pack everything for download (appears under Output)
import shutil
shutil.make_archive('/kaggle/working/results_gpu_first_round', 'zip', 'results')
print(open('results/summary_size128_azure.txt').read())